In [ ]:
import torch 
import torch.nn as nn 
import torch.optim as optim 
from torch.optim import AdamW
from PIL import Image
from torch.utils.data import DataLoader, Dataset 
from torchvision import datasets, transforms, models 
from transformers import DistilBertModel, DistilBertTokenizer
import numpy as np 
import re
import os 

import matplotlib.pyplot as plt 
from sklearn.metrics import confusion_matrix 
import seaborn as sns 

In [ ]:
def read_text_files_with_labels(path):
    texts = []
    labels = []
    fn_arr = [] 
    class_folders = sorted(sorted(os.listdir(path)))
    label_map = {class_name: idx for idx, class_name in enumerate(class_folders)}

    for class_name in class_folders:
        class_path = os.path.join(path,class_name)
        if os.path.isdir(class_path):
            file_names = os.listdir(class_path)
            for file_name in file_names:
                file_path = os.path.join(class_path, file_name)
                if os.path.isfile(file_path):
                    file_name_no_ext, _ = os.path.splitext(file_name)
                    text = file_name_no_ext.replace('_',' ')
                    text_without_digits = re.sub(r'\d+','',text)
                    texts.append(text_without_digits)
                    labels.append(label_map[class_name])
                    fn_arr.append(os.path.join(class_path,file_name))

    return np.array(texts), np.array(labels), np.array(fn_arr)

class CustomDataset(Dataset):
    def __init__(self, dir, fns, texts, labels, tokenizer, max_len, transform):
        self.texts = texts
        self.labels = labels
        self.tokenizer = tokenizer
        self.max_len = max_len
        self.dir = dir 
        self.fns = fns 
        self.transform = transform 

    def __len__(self):
        # print('Grabbing length...')
        return len(self.texts)

    def __getitem__(self, idx):
        # print('Grabbing item...')
        text = str(self.texts[idx])
        label = self.labels[idx]
        im_path = self.fns[idx]
        image = Image.open(im_path).convert('RGB')

        image = self.transform(image)

        encoding = self.tokenizer(
            text,
            add_special_tokens=True,
            max_length=self.max_len,
            padding='max_length',
            truncation=True,
            return_attention_mask=True,
            return_tensors='pt'
        )

        return {
            'text': text,
            'input_ids': encoding['input_ids'].flatten(),
            'attention_mask': encoding['attention_mask'].flatten(),
            'label': torch.tensor(label, dtype=torch.long),
            'image': image
        }

In [ ]:
class MMM(nn.Module):
    def __init__(self):
        super(MMM,self).__init__() 

        self.text_model = DistilBertModel.from_pretrained('distilbert-base-uncased')
        self.image_model = models.mobilenet_v2(pretrained=True)
        # self.text_model.eval()
        # self.image_model.eval() 
        for param in self.text_model.parameters():
            param.requires_grad = False 
        for param in self.image_model.features.parameters():
            param.requires_grad = False 
        for param in self.image_model.features[-2:].parameters():
            param.requires_grad = True
        for layer in self.text_model.transformer.layer[-2:]:
            for param in layer.parameters():
                param.requires_grad = True

        # Remove image classifier 
        self.image_num_features = self.image_model.classifier[1].in_features 
        self.image_model.classifier = nn.Identity() 
        # Do not need to remove classifier for DistilBERT because it does not have a head on the base model! 
        self.text_num_features = self.text_model.config.hidden_size 
        self.text_proj = nn.Linear(self.text_num_features,256)
        self.image_proj = nn.Linear(self.image_num_features,256)
        self.fc = nn.Sequential(
            nn.Linear(512, 256), # Example dimensions
            nn.ReLU(),
            nn.Dropout(0.5),
            nn.Linear(256, 4) # Final output layer
        )
    
    def forward(self,input_ids,attention_mask, images):
        text_output = self.text_model(input_ids=input_ids,attention_mask=attention_mask)
        text_embeddings = text_output.last_hidden_state[:,0,:]
        text_embeddings = self.text_proj(text_embeddings)
        text_embeddings = nn.functional.normalize(text_embeddings,p=2,dim=1)

        image_embeddings = self.image_model(images)
        image_embeddings = self.image_proj(image_embeddings)
        image_embeddings = nn.functional.normalize(image_embeddings,p=2,dim=1)
        # image_embeddings = self.flatten(image_embeddings)
        combined = torch.cat((text_embeddings,image_embeddings),dim=1)
        
        output = self.fc(combined)

        return output 


In [ ]:
tokenizer = DistilBertTokenizer.from_pretrained('distilbert-base-uncased')

train_dir = "CVPR_2024_datasetTrain"
val_dir = "CVPR_2024_datasetVal"
test_dir = "CVPR_2024_datasetTest"

text_train,labels_train,fns_train = read_text_files_with_labels(train_dir)
text_val,labels_val,fns_val = read_text_files_with_labels(val_dir)
text_test,labels_test,fns_test = read_text_files_with_labels(test_dir) 

transform = transforms.Compose([transforms.Resize((224,224)),
                                transforms.RandomHorizontalFlip(),
                                transforms.ToTensor(),
                                transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.255])])
max_len = 24 
train_dataset = CustomDataset(train_dir, fns_train, text_train, labels_train, tokenizer, max_len, transform) 
val_dataset = CustomDataset(val_dir, fns_val, text_val, labels_val, tokenizer, max_len, transform) 
test_dataset = CustomDataset(test_dir, fns_test, text_test, labels_test, tokenizer, max_len, transform) 

train_dataloader = DataLoader(train_dataset, batch_size=8, shuffle=True, num_workers=0)
val_dataloader = DataLoader(val_dataset, batch_size=8, shuffle=False, num_workers=0)
test_dataloader = DataLoader(test_dataset, batch_size=8, shuffle=False, num_workers=0)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu') 

model = MMM() 
model = model.to(device)
# from sklearn.utils.class_weight import compute_class_weight

# class_weights = compute_class_weight(
#     class_weight='balanced',
#     classes=np.unique(labels_train),
#     y=labels_train
# )

# class_weights = torch.tensor(class_weights, dtype=torch.float).to(device)
# criterion = nn.CrossEntropyLoss(weight=class_weights)
criterion = nn.CrossEntropyLoss() 
optimizer = AdamW([
    {"params": model.text_model.parameters(), "lr": 2e-5},
    {"params": model.image_model.parameters(), "lr": 1e-4},
    {"params": model.fc.parameters(), "lr": 2e-3}
])
print(fns_train[0])

In [ ]:
def train(model, iterator, optimizer, criterion, device):
    # print(f'Set model training mode...')
    model.train()
    total_loss = 0
    batch_num = 0 
    # print(f'Entering batch...')
    for batch in iterator:
        batch_num += 1
        # print(f'Batch num: {batch_num}')
        input_ids = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        images = batch['image'].to(device) 
        labels = batch['label'].to(device) 
        # print(f'Grabbed inputs...')
        optimizer.zero_grad()
        with torch.set_grad_enabled(True):
            output = model(input_ids, attention_mask, images) 
            print(f'Made model prediction...')
            loss = criterion(output, labels)
            # print(f'Found loss...')
            loss.backward()
            # print(f'Calculated backprop...')
            optimizer.step()
            # print(f'Stepped optimizer...')

        total_loss += loss.item()
        

    return total_loss / len(iterator)

In [ ]:
def evaluate(model,iterator,criterion,device):
    model.eval()
    total_loss = 0
    with torch.no_grad():
        for batch in iterator:
            input_ids = batch['input_ids'].to(device)
            attention_mask = batch['attention_mask'].to(device)
            labels = batch['label'].to(device)
            images = batch['image'].to(device) 

            output = model(input_ids, attention_mask, images)
            print(f'Made model prediction...')
            loss = criterion(output, labels)

            total_loss += loss.item() 

    return total_loss / len(iterator)

In [ ]:
def predict(model,dataloader,device):
    model.eval()
    predictions = []
    with torch.no_grad():
        for batch in dataloader:
            input_ids = batch['input_ids'].to(device)
            attention_mask = batch['attention_mask'].to(device)
            images = batch['image'].to(device) 

            outputs = model(input_ids,attention_mask,images)
            print(f'Made model prediction...')
            _, preds = torch.max(outputs, dim=1)

            predictions.extend(preds.cpu().numpy())

    return predictions 

In [ ]:
EPOCHS = 4 
best_loss = 1e+10
for epoch in range(EPOCHS):
    print(f'Epoch {epoch+1}/{EPOCHS}')
    train_loss = train(model,train_dataloader,optimizer,criterion,device)
    print(f'Epoch {epoch+1}, Train Loss: {train_loss:.4f}')
    val_loss = evaluate(model,val_dataloader,criterion,device)
    print(f'Epoch: {epoch+1}, Val Loss: {val_loss:.4f}')
    if val_loss < best_loss:
        best_loss = val_loss
        torch.save(model.state_dict(), 'best_model_text.pth')

In [ ]:
# model.load_state_dict(torch.load('best_model_text.pth'))
test_predictions = predict(model,test_dataloader,device)
print(f'Accuracy: {(test_predictions == labels_test).sum()/labels_test.size:.4f}')

cm = confusion_matrix(labels_test, test_predictions) 

plt.figure(figsize=(8,6))
sns.heatmap(cm,annot=True,cmap='Blues',fmt='g',cbar=False)

plt.title('Confusion Matrix')
plt.xlabel('Predicted Labels')
plt.ylabel('True Labels')
plt.show() 